# Create a Foundry IQ Agent via Python

In this notebook, you will create a **Foundry IQ agent** programmatically using the Python SDK. The portal workflow builds the same agent by clicking through the Azure AI Foundry UI; this notebook gives you the code-first alternative — useful for automation, reproducibility, and integrating agent creation into CI/CD pipelines.

**Prerequisites:**
- An existing Foundry IQ **knowledge base** backed by an Azure AI Search index. The default `.env` values assume a knowledge base named `support-kb` built over a container of support documents. If your knowledge base has a different name, set `KNOWLEDGE_BASE_NAME` and `SEARCH_SERVICE_ENDPOINT` in `.env` accordingly — this notebook only needs the knowledge base name and the Search endpoint, not the underlying container layout.
- A deployed model in your Foundry project (e.g., `gpt-5.1`)
- Azure CLI installed and authenticated (`az login`)

This notebook is **self-contained**: it creates the `RemoteTool` connection from your Foundry project to the knowledge base MCP endpoint, so you do not need to set that connection up in advance.

**What you will do:**
1. Connect to your Foundry project
2. Create and verify the `RemoteTool` connection to the knowledge base
3. Configure an MCP tool pointing to your knowledge base
4. Create an agent with grounding instructions
5. Chat with the agent and verify it answers only from the knowledge base

### Step 1: Install Packages

Run the cell below to install every Python library this notebook needs.
You only need to do this once per environment.

In [ ]:
# Azure AI Projects SDK, OpenAI client, dotenv for config, and Azure Identity for authentication
# NOTE: Use Python 3.12 — Python 3.14 has typing incompatibilities with the openai package
%pip install azure-ai-projects==2.2.0 openai==2.9.0 python-dotenv azure-identity

### Step 2: Load Configuration

We read all connection details from a `.env` file so that secrets never appear in source code. We also build the MCP endpoint URL that points to the knowledge base.

> **Fill in `.env` first.** Open the `.env` file in this folder and replace every `<...>` placeholder with your own value before running this cell. Entries that already have a value have working defaults — leave them as they are.

In [ ]:
# -- Standard library --
import os

# -- Third-party helpers --
from dotenv import load_dotenv                      # Reads .env files into os.environ
from azure.identity import DefaultAzureCredential   # Handles Azure authentication
from azure.ai.projects import AIProjectClient       # Foundry project SDK
from azure.ai.projects.models import (
    PromptAgentDefinition,   # Describes the agent's model, instructions, and tools
    MCPTool,                 # Represents an MCP server the agent can call
)

# Load environment variables from the .env file in the current directory.
# override=True so that editing .env and re-running this cell picks up the new
# values instead of keeping stale ones already loaded into the kernel.
load_dotenv(override=True)

# -- Read each value from the environment --
my_endpoint         = os.getenv("FOUNDRY_PROJECT_ENDPOINT")                # Foundry project URL
project_resource_id = os.getenv("FOUNDRY_PROJECT_RESOURCE_ID")             # Full ARM resource ID of the project (used to create the connection)
my_model            = os.getenv("MODEL_DEPLOYMENT_NAME")                   # Deployed model name
search_endpoint     = os.getenv("SEARCH_SERVICE_ENDPOINT")                 # Azure AI Search endpoint
kb_name             = os.getenv("KNOWLEDGE_BASE_NAME")                     # Name of the knowledge base
search_conn_name    = os.getenv("SEARCH_SERVICE_PROJECT_CONNECTION_NAME")  # Connection name in Foundry

# Fail fast with a clear message if a required value is missing or still a
# placeholder. Without this guard a missing value surfaces much later as a
# cryptic SDK or HTTP error deep inside agent/connection creation.
required = {
    "FOUNDRY_PROJECT_ENDPOINT": my_endpoint,
    "FOUNDRY_PROJECT_RESOURCE_ID": project_resource_id,
    "MODEL_DEPLOYMENT_NAME": my_model,
    "SEARCH_SERVICE_ENDPOINT": search_endpoint,
    "KNOWLEDGE_BASE_NAME": kb_name,
    "SEARCH_SERVICE_PROJECT_CONNECTION_NAME": search_conn_name,
}
missing = [key for key, value in required.items() if not value or "<" in value]
if missing:
    raise ValueError(
        "Missing required configuration: " + ", ".join(missing) + ".\n"
        "Copy .env.example to .env and replace every <...> placeholder with your "
        "own values before running this cell."
    )

# Build the full MCP URL that points to the knowledge base
# This URL follows the Foundry IQ pattern: {search_endpoint}/knowledgebases/{kb_name}/mcp
kb_mcp_url = (
    f"{search_endpoint}/knowledgebases/{kb_name}"
    f"/mcp?api-version=2025-11-01-preview"
)

print(f"Foundry endpoint : {my_endpoint}")
print(f"Model            : {my_model}")
print(f"Knowledge base   : {kb_name}")
print(f"MCP URL          : {kb_mcp_url}")

### Step 3: Connect to Foundry

The `AIProjectClient` is the main entry point for managing agents,
connections, and other resources inside your Foundry project. The
`chat_client` gives us an OpenAI-compatible interface for conversations.

In [ ]:
# Create the Foundry project client using default Azure credentials
# DefaultAzureCredential() automatically picks up your Azure CLI login or managed identity
# allow_preview=True turns on Foundry preview features. Foundry IQ runs on a preview
# API surface, so we set it for forward compatibility with knowledge-base (MCP) features.
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

# Obtain an OpenAI-compatible client from the Foundry project
chat_client = foundry_client.get_openai_client()

print("Connected to Foundry.")

### Step 4: Create the MCP Connection and Grant Access

The agent reaches the knowledge base through an MCP (Model Context Protocol) endpoint, and Foundry needs a **`RemoteTool` connection** that authenticates to that endpoint using the project's managed identity. We do two things here:

1. Create the connection with a management-plane `PUT`.
2. Grant the project's managed identity the **`Search Index Data Reader`** role on the search service, so that identity can actually read the knowledge base.

Then we confirm the connection is visible in the project.

> The connection category must be `RemoteTool` with `ProjectManagedIdentity` auth — a regular `CognitiveSearch` connection will not work for the MCP endpoint. The project identity also needs `Search Index Data Reader` on the search service, or the agent returns **HTTP 403** when it enumerates the knowledge base tools. Re-running these cells is safe: the connection `PUT` updates in place, and the role assignment returns `409` if it already exists.

In [ ]:
# Create the RemoteTool connection from the Foundry project to the knowledge base MCP endpoint.
# This uses the management plane (ARM) and configures the connection to authenticate with the
# project's managed identity — no API keys are stored on the connection.
import requests
from azure.identity import get_bearer_token_provider

if not project_resource_id:
    raise ValueError(
        "Set FOUNDRY_PROJECT_RESOURCE_ID in .env before creating the MCP connection. "
        "Find it in the Azure Portal on the Foundry project's Properties page."
    )

# Data-plane MCP endpoint the connection points at (the knowledge base built earlier)
mcp_endpoint = (
    f"{search_endpoint.rstrip('/')}/knowledgebases/{kb_name}"
    f"/mcp?api-version=2025-11-01-preview"
)

# Acquire a management-plane bearer token from the same credential used everywhere else
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://management.azure.com/.default"
)

create_response = requests.put(
    f"https://management.azure.com{project_resource_id}/connections/{search_conn_name}"
    f"?api-version=2025-10-01-preview",
    headers={"Authorization": f"Bearer {token_provider()}"},
    json={
        "name": search_conn_name,
        "properties": {
            "authType": "ProjectManagedIdentity",
            "category": "RemoteTool",
            "target": mcp_endpoint,
            "isSharedToAll": True,
            "audience": "https://search.azure.com/",
            "metadata": {"ApiType": "Azure"},
        },
    },
)
create_response.raise_for_status()
print(f"RemoteTool connection '{search_conn_name}' is ready.")

In [ ]:
# Grant the Foundry project's managed identity read access to the search service.
# The RemoteTool connection authenticates to the knowledge base MCP endpoint AS the
# project's managed identity, so that identity needs the "Search Index Data Reader"
# role on the search service. Without this grant the agent gets HTTP 403 when it tries
# to enumerate the knowledge base tools.
import uuid

# Parse the subscription and the project's resource group from the project resource ID
_parts = project_resource_id.strip("/").split("/")
sub_id     = _parts[1]
project_rg = _parts[3]

# The search service may live in its own resource group; default to the project's RG
search_rg = os.getenv("SEARCH_SERVICE_RESOURCE_GROUP", project_rg)

# Derive the search service name from its endpoint (https://<name>.search.windows.net)
search_service_name = search_endpoint.split("//", 1)[-1].split(".", 1)[0]

search_resource_scope = (
    f"/subscriptions/{sub_id}/resourceGroups/{search_rg}"
    f"/providers/Microsoft.Search/searchServices/{search_service_name}"
)

# Resolve the project's managed identity principal ID from the project resource
proj = requests.get(
    f"https://management.azure.com{project_resource_id}?api-version=2025-06-01",
    headers={"Authorization": f"Bearer {token_provider()}"},
)
proj.raise_for_status()
project_principal_id = proj.json()["identity"]["principalId"]

# "Search Index Data Reader" built-in role definition (well-known role GUID)
SEARCH_INDEX_DATA_READER = "1407120a-92aa-4202-b7e9-c0e197c71c8f"
role_def_id = (
    f"/subscriptions/{sub_id}/providers/Microsoft.Authorization"
    f"/roleDefinitions/{SEARCH_INDEX_DATA_READER}"
)

role_assignment = requests.put(
    f"https://management.azure.com{search_resource_scope}"
    f"/providers/Microsoft.Authorization/roleAssignments/{uuid.uuid4()}"
    f"?api-version=2022-04-01",
    headers={"Authorization": f"Bearer {token_provider()}"},
    json={
        "properties": {
            "roleDefinitionId": role_def_id,
            "principalId": project_principal_id,
            "principalType": "ServicePrincipal",
        }
    },
)

if role_assignment.status_code in (200, 201):
    print(f"Granted 'Search Index Data Reader' to the project identity on '{search_service_name}'.")
elif role_assignment.status_code == 409:
    print(f"Project identity already has access to '{search_service_name}'.")
else:
    role_assignment.raise_for_status()

print("RBAC can take 1-2 minutes to propagate. If the agent returns a 403 below, wait and retry.")

In [ ]:
# Verify the connection exists in the project
found = False
for conn in foundry_client.connections.list():
    if conn.name == search_conn_name:
        found = True
        print(f"Connection found: {conn.name}")
        break

if not found:
    print(f"WARNING: Connection '{search_conn_name}' not found in the project.")
    print("Available connections:")
    for c in foundry_client.connections.list():
        print(f"  - {c.name}")

### Step 5: Configure the Knowledge Base Tool

An `MCPTool` tells the agent how to reach the knowledge base. Foundry IQ
exposes each knowledge base as an MCP (Model Context Protocol) endpoint.
The agent will call this tool automatically whenever it needs to retrieve
information from the indexed documents.

In [ ]:
# Create the MCP tool that connects the agent to the knowledge base
kb_tool = MCPTool(
    server_label="knowledge-base",                # Human-readable label shown in traces
    server_url=kb_mcp_url,                        # Full URL of the knowledge base MCP endpoint
    require_approval="never",                     # Auto-approve every tool call (no human-in-the-loop)
    allowed_tools=["knowledge_base_retrieve"],    # Only expose the retrieval tool
    project_connection_id=search_conn_name,       # Pass the connection NAME (not ID) from .env
)

print("Knowledge base MCP tool configured.")

### Step 6: Define the Agent Behavior

The system prompt tells the agent to answer **only** from the knowledge
base. If the answer is not in the indexed documents, the agent should
say so rather than falling back to its training data.

In [ ]:
# System prompt that controls how the agent responds
agent_instructions = """
You are a knowledgeable support assistant. Your job is to answer questions
by retrieving information from the knowledge base.

Rules:
1. Always use the knowledge base to find your answers. Do not answer from
   your own training data.
2. When you find relevant information, quote the source and provide a
   clear, specific answer.
3. If the knowledge base does not contain the answer, say so clearly
   rather than guessing.
4. Keep your answers concise and actionable.
"""

### Step 7: Create the Agent

We register a new agent version in Foundry. The agent uses the deployed
model, follows the behavior instructions, and has the knowledge base
tool attached so it can query the indexed documents.

In [ ]:
# Choose a name for the agent — this is how we reference it in conversations
kb_agent_name = "iq-support-agent"

# Create (or update the version of) the agent in Foundry
kb_agent = foundry_client.agents.create_version(
    agent_name=kb_agent_name,
    definition=PromptAgentDefinition(
        model=my_model,                # The deployed model powering the agent
        instructions=agent_instructions,  # System prompt defining agent behavior
        tools=[kb_tool],               # Give it access to the knowledge base
    ),
)

print(f"Agent created — ID: {kb_agent.id}, Version: {kb_agent.version}")

### Step 8: Start a Conversation and Ask a Question

A conversation holds the message history between the user and the agent.
We create a session, then send a question. The agent will search the
knowledge base, retrieve relevant documents, and compose a grounded answer.

In [ ]:
# Open a new conversation session
chat_session = chat_client.conversations.create()
print(f"Conversation started — ID: {chat_session.id}")

# Ask the agent a question about the knowledge base content
question = "What is the warranty policy for electronic devices?"

# Send the question to the agent and wait for the response
response = chat_client.responses.create(
    conversation=chat_session.id,                   # Link to the conversation session
    extra_body={
        "agent_reference": {
            "name": kb_agent_name,                  # Which agent handles this request
            "type": "agent_reference",
        }
    },
    input=question,                                 # The actual question
)

print(f"Agent:\n\n{response.output_text}")

### Step 9: Ask a Follow-Up Question

Because we are using the same conversation session, the agent retains
context from the previous exchange. This demonstrates that the
conversation history carries forward across turns.

In [ ]:
# Ask a follow-up question in the same conversation
followup = "What are the steps to troubleshoot a device that will not power on?"

followup_response = chat_client.responses.create(
    conversation=chat_session.id,
    extra_body={
        "agent_reference": {
            "name": kb_agent_name,
            "type": "agent_reference",
        }
    },
    input=followup,
)

print(f"Agent:\n\n{followup_response.output_text}")

### Step 10: Test the Grounding Boundary

This is the most important test. We ask a question that is clearly
**not** in the knowledge base. A properly grounded agent should decline
to answer from its training data and tell us the information is not
available in the indexed documents.

In [ ]:
# Ask something that is definitely NOT in the knowledge base
out_of_scope = "What is the current stock price of Microsoft?"

boundary_response = chat_client.responses.create(
    conversation=chat_session.id,
    extra_body={
        "agent_reference": {
            "name": kb_agent_name,
            "type": "agent_reference",
        }
    },
    input=out_of_scope,
)

print(f"Agent (out of scope):\n\n{boundary_response.output_text}")

---

### Summary

In this notebook, you created a Foundry IQ agent entirely through code:

1. **Connected** to a Foundry project using `AIProjectClient`
2. **Configured** an `MCPTool` pointing to an existing knowledge base
3. **Created** an agent version with grounding instructions and the knowledge base tool
4. **Tested** the agent with in-scope questions, follow-ups, and an out-of-scope boundary check

The agent is now grounded on the knowledge base and will only answer from
the indexed documents. In production, update the knowledge base content and
the agent automatically picks up new information on the next retrieval —
no agent redeployment needed.

This same pattern works for any Foundry IQ knowledge base. Swap out the
`.env` values to point at a different knowledge base and model, and the
rest of the code stays the same.